In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
from ase.io.trajectory import Trajectory
from flonacomldft.collective_variables import get_collective_variables
import numpy as np

In [3]:
# path_database = '/mnt/home/amolina/ceph/project-database/andersen/'
# md_files = ['trajectories/27037382_is0_andersen.traj',  'trajectories/27037385_is1_andersen.traj']
# 
# trajs = {i: Trajectory(path_database + md_files[i]) for i in range(len(md_files))}

In [4]:
#temperature = {i: np.array([molecule.get_temperature() for molecule in trajs[i]]) for i in trajs.keys()}

In [5]:
#u = {i: np.array([molecule.get_potential_energy() for molecule in trajs[i]]) for i in trajs.keys()}

In [6]:
#cv = {i: np.stack([get_collective_variables(molecule) for molecule in trajs[i][:5]]) for i in trajs.keys()}

In [7]:
from flonacomldft.utils.io_utils import get_path, load_pickle_file
import torch

In [8]:
flow_models = [ load_pickle_file('dict_flow_model_is'+str(i)+'.pkl', get_path() + '/andersen/models/')['model'] for i in range(2)]
mlp_model = [load_pickle_file('dict_mlp_model_is'+str(i)+'.pkl', get_path() + '/andersen/models/')['model'] for i in range(2)]

In [9]:
from flonacomldft.models.mixture import Mixture

flow_model = Mixture(flow_models, torch.tensor([0.5, 0.5]))

In [10]:
n_chains = 10
n_steps = 20
xs_init = flow_model.sample(n_chains)
us_init = mlp_model[0](xs_init)
isomers_init = torch.zeros(n_chains, 1)

init = torch.cat([xs_init, us_init, isomers_init], dim=1)

In [11]:
from flonacomldft.sampling import run_metropolis

In [12]:
run_metropolis(model = flow_model, 
                init = init, 
                n_chains = n_chains,
                n_steps = n_steps,
                id_run = 'N/A', 
                energy_type = 'mlp', 
                temperature = 350,
                mixture = True,
                mlp_models = mlp_model,
                frac_computed = 0.2,
                dim = 12,
                update_weights = True,
                scheduler_weights = 5,
                alpha = 0.5,
                return_ratios = False,
                return_proposals = True,
                with_tqdm = False,
                folder_name = None,
                )

Running Metropolis-Hastings
Number of chains: 10
Number of steps: 20
Temperature: 350K
Energy Type: mlp
Mixture model
Use Neural Predictor: True
Mixture Model: True
Step 	 Acc Rate 	 Population
0 	 0.600 		 0.300
1 	 0.400 		 0.300
2 	 0.300 		 0.200
3 	 0.300 		 0.200
4 	 0.400 		 0.100
5 	 0.200 		 0.000
6 	 0.400 		 0.000
7 	 0.000 		 0.000
8 	 0.300 		 0.000
9 	 0.200 		 0.000
10 	 0.300 		 0.000
11 	 0.400 		 0.000
12 	 0.200 		 0.000
13 	 0.200 		 0.000
14 	 0.400 		 0.000
15 	 0.200 		 0.000
16 	 0.300 		 0.000
17 	 0.300 		 0.000
18 	 0.000 		 0.000
19 	 0.000 		 0.000


{'xs': [tensor([[ 2.0951e-01, -4.7896e-02, -1.2518e-01, -4.0803e-02, -2.7033e-02,
            2.9284e-02, -1.0204e-01,  1.1538e-01,  1.2519e-02, -6.1654e-01,
            7.4691e-01,  8.9166e-01],
          [ 1.7120e-01, -9.3339e-02, -1.0588e-01, -5.4286e-02,  6.7790e-02,
            6.5091e-02, -6.8336e-02,  4.4105e-02, -1.9826e-02, -5.9376e-01,
            9.7829e-01,  3.8664e-01],
          [-4.9108e-02, -1.1589e-01, -1.0551e-01, -9.8167e-02,  3.4588e-02,
            1.4554e-01,  5.4703e-02,  5.4388e-02,  3.8420e-03, -1.2027e-01,
            8.4034e-01,  1.0973e-01],
          [ 1.4346e-01,  8.1375e-02,  1.0783e-01, -1.1247e-02,  1.0182e-01,
           -9.9779e-02, -4.6252e-02, -8.0385e-02, -3.7771e-02,  2.6280e-01,
            1.3481e+00,  1.8151e-02],
          [ 9.2860e-03,  1.2678e-02,  8.7172e-02, -8.9720e-02,  1.1166e-01,
            3.4998e-02,  3.3101e-02, -5.6621e-02, -2.9016e-02,  2.3713e-01,
            5.4354e-02, -8.4855e-02],
          [-1.5579e-02,  2.7573e-02,  4.5488

In [13]:
from flonacomldft.utils.io_utils import load_csv_file
from flonacomldft.train_flow_from_data import train_flow
from flonacomldft.train_mlp_from_data import train_mlp

In [19]:
import copy
import time

def Transpose(x):
    return x.permute(*torch.arange(x.ndim - 1, -1, -1))

def adaptive_sampling(
    mcmc_init,
    n_chains,
    n_steps,
    n_runs,
    flow_init_train,
    flow_init_test,
    dict_flows_init,
    flow_hyperparams,
    energy_type,
    temperature,
    mixture,
    dim=12,
    dict_mlps_init=None,
    mlp_init_train=None,
    mlp_init_test=None,
    mlp_hyperparams=None,
    train_mlps=True,
    frac_computed=0.2,
    init_weights=None, 
    update_weights=True,
    scheduler_weights=10,
    alpha=0.5,
    n_samples_train_flow=None,
    folder_name=None,
    device='cpu', 
    ):

    print('Adaptive sampling')
    print('Number of runs: ', n_runs)
    print('Number of chains: ', n_chains)
    print('Number of steps: ', n_steps)
    print('Temperature: ', temperature)
    print('Energy type: ', energy_type)
    print('Mixture model: ', mixture)

    isomer_labels = torch.unique(torch.cat(flow_init_train)[:, dim+1]).int()
    n_isomers = isomer_labels.shape[0]

    print('Isomer labels: ', isomer_labels.tolist())

    xs_for_flows_train = [ flow_init_train[i] for i in range(n_isomers) ]
    xs_for_flows_test = [ flow_init_test[i] for i in range(n_isomers) ]

    print('Flow train dataset shapes: ', [ list(xs_for_flows_train[i].shape) for i in range(n_isomers) ])
    print('Flow test dataset shapes: ', [ list(xs_for_flows_test[i].shape) for i in range(n_isomers) ])

    if ('mlp' in energy_type) and (dict_mlps_init is not None):

        if train_mlps:

            xs_for_mlps_train = [ mlp_init_train[i] for i in range(n_isomers) ]
            xs_for_mlps_test = [ mlp_init_test[i] for i in range(n_isomers) ]

            n_samples_train_mlp = torch.tensor( [ xs_for_mlps_train[i].shape[0] for i in range(n_isomers) ] )
            mlp_batch_size = torch.tensor( [ mlp_hyperparams[i]['batch_size'] for i in range(n_isomers) ] )

            fix_iters_per_batch = torch.ceil(n_samples_train_mlp / mlp_batch_size).int()

            print(n_samples_train_mlp)
            print(mlp_batch_size)
            print(fix_iters_per_batch)

            dict_mlps = [ copy.deepcopy(dict_mlps_init) ]

            print('MLP train dataset shapes: ', [ list(xs_for_mlps_train[i].shape) for i in range(n_isomers) ])
            print('MLP test dataset shapes: ', [ list(xs_for_mlps_test[i].shape) for i in range(n_isomers) ])

        else:

            print('MLPs will not be trained')


    dict_flows = [ copy.deepcopy(dict_flows_init) ]

    xs_proposals = []
    us_proposals = []
    isomers_proposals = []

    if mixture and init_weights is None:
        
        init_weights = torch.tensor([1/n_isomers for i in range(n_isomers)]).detach()
        print('Initial weights: ', init_weights.tolist())
    
    elif mixture and init_weights is not None:
        
        print('Initial weights: ', init_weights.tolist())

    if n_samples_train_flow is None:
        n_samples_train_flow = torch.tensor( [ xs_for_flows_train[i].shape[0] for i in range(n_isomers) ] )
    
    print('Number of samples for training flows: ', n_samples_train_flow)
    
    if ("dft" in energy_type) or ("emt" in energy_type):

        use_calc = True

        xs_calc = []
        us_calc = []
        isomers_calc = []
        inds_calc = []

    else:

        use_calc = False

    xs = []
    us = []
    accs = []
    isomers = []
    time_mcmc = []

    time_step_flow = []
    time_step_adaptive = []

    init = mcmc_init

    for i in range(n_runs):

        if mixture:
            flow_models = [ dict_flows[i][j]['model'] for j in range(n_isomers)]
            model = Mixture(flow_models, init_weights)
            print('Current weights: ', model.weights.tolist())
        else:
            model = dict_flows[i][0]['model']

        if ('mlp' in energy_type) and (train_mlps == True) and (dict_mlps_init is not None):
            mlp_models = [ dict_mlps[i][j]['model'] for j in range(n_isomers)]

        elif ('mlp' in energy_type) and (train_mlps == False) and (dict_mlps_init is not None):
            mlp_models = [ dict_mlps_init[i]['model'] for i in range(n_isomers)]

        mcmc = run_metropolis(model = model, 
                                init = init, 
                                n_chains = n_chains,
                                n_steps = n_steps,
                                id_run = i, 
                                energy_type = energy_type, 
                                temperature = temperature,
                                mixture = mixture,
                                mlp_models = mlp_models,
                                frac_computed = frac_computed,
                                dim = dim,
                                update_weights = update_weights,
                                scheduler_weights = scheduler_weights,
                                alpha = alpha,
                                return_ratios = False,
                                return_proposals = True,
                                with_tqdm = False,
                                folder_name = folder_name + '/DFTAdaptive',
                                device=device,
                                )
        
        time_step_adaptive.append(time.time())

        xs.append(mcmc['xs'])
        us.append(mcmc['us'])
        accs.append(mcmc['accs'])
        isomers.append(mcmc['isomers'])
        time_mcmc.append(mcmc['time_mcmc'])

        xs_proposals.append(mcmc['xs_proposals'])
        us_proposals.append(mcmc['us_proposals'])
        isomers_proposals.append(mcmc['isomers_proposals'])

        if use_calc:

            xs_calc.append(mcmc['xs_calc'])
            us_calc.append(mcmc['us_calc'])
            isomers_calc.append(mcmc['isomers_calc'])
            inds_calc.append(mcmc['inds_calc'])

        init = torch.cat( (xs[i][-1].clone(),
            us[i][-1].clone().reshape(-1, 1),                 
            isomers[i][-1].clone().reshape(-1, 1)), 
            dim=1)
        
        chains_flatten = Transpose(
            torch.cat(
                (
                    Transpose(torch.stack(xs[i]).clone()),
                    Transpose(torch.stack(us[i]).clone().reshape(n_steps, n_chains, 1)),
                    Transpose(torch.stack(isomers[i]).clone().reshape(n_steps, n_chains, 1)),
                ),
                dim=0,
            )
        )

        chains_flatten = chains_flatten.reshape(n_steps * n_chains, 
                                                chains_flatten.shape[-1])
        
        mask_flow = chains_flatten[:, -1]

        dict_new_flows = []

        if train_mlps:

            dict_new_mlps = []

            xs_calc_run = torch.stack(mcmc['xs_calc'])
            us_calc_run = torch.stack(mcmc['us_calc'])
            isomers_calc_run = torch.stack(mcmc['isomers_calc'])

            configs_dft_flatten = Transpose(
                torch.cat(
                        (
                            Transpose(xs_calc_run.clone()),
                            us_calc_run.clone().reshape(1, -1),
                            isomers_calc_run.reshape(1, -1),
                        ),
                        dim=0,
                    )
                )

            mask_mlp = configs_dft_flatten[:, -1]


        for mode in range(n_isomers):

            print('Isomer: ', isomer_labels[mode].item())

            xs_from_chains = chains_flatten.clone()[mask_flow == isomer_labels[mode]].clone()

            print('Flow train dataset shape: ', list(xs_for_flows_train[mode].shape))
            print('Flow chains dataset shape: ', list(xs_from_chains.shape))

            xs_for_flows_train[mode] = torch.cat(
                    (xs_for_flows_train[mode].clone(), xs_from_chains.clone())
                )
            
            print('Flow train dataset shape: ', list(xs_for_flows_train[mode].shape))

            print('Flow input train dataset shape: ', list(xs_for_flows_train[mode][-n_samples_train_flow[mode]:].shape) )
        
            flow_model = copy.deepcopy(dict_flows[i][mode]['model'])

            dict_new_flow = train_flow(
                model=flow_model,
                train=xs_for_flows_train[mode][-n_samples_train_flow[mode]:],
                test=xs_for_flows_test[mode],
                **flow_hyperparams[mode],
                dim=dim,
                mlp_model=mlp_models[mode],
            )

            time_step_flow.append(time.time())

            dict_new_flows.append(dict_new_flow)

            if train_mlps:

                mask_mlp_mode = mask_mlp == isomer_labels[mode]

                indexes = torch.randperm(configs_dft_flatten[mask_mlp_mode].shape[0])

                split_ratio = 0.8
                split_index = int(configs_dft_flatten[mask_mlp_mode].shape[0] * split_ratio)
                configs_dft_flatten_train = configs_dft_flatten[mask_mlp_mode][indexes[:split_index]]
                configs_dft_flatten_test = configs_dft_flatten[mask_mlp_mode][indexes[split_index:]]

                print('MLP chains train dataset shape: ', list(configs_dft_flatten_train.shape))
                print('MLP chains test dataset shape: ', list(configs_dft_flatten_test.shape))

                xs_for_mlps_train[mode] = torch.cat(
                        (xs_for_mlps_train[mode].clone(), 
                         configs_dft_flatten_train.clone())
                    )
                
                xs_for_mlps_test[mode] = torch.cat(
                        (xs_for_mlps_test[mode].clone(), 
                         configs_dft_flatten_test.clone())
                    )
                
                print('MLP train dataset shape: ', list(xs_for_mlps_train[mode].shape))
                print('MLP test dataset shape: ', list(xs_for_mlps_test[mode].shape))

                new_batch_size = torch.ceil(xs_for_mlps_train[mode].shape[0] / fix_iters_per_batch[mode]).int().item()
                mlp_hyperparams[mode]['batch_size'] = new_batch_size

                print("Number of gradient steps per epoch: ", xs_for_mlps_train[mode].shape[0] / mlp_hyperparams[mode]['batch_size'], 
                      mlp_hyperparams[mode]['n_iter'], (xs_for_mlps_train[mode].shape[0] / mlp_hyperparams[mode]['batch_size']) * mlp_hyperparams[mode]['n_iter'] )

                dict_new_mlp = train_mlp(
                    model=mlp_models[mode],
                    train=xs_for_mlps_train[mode],
                    test=xs_for_mlps_test[mode],
                    **mlp_hyperparams[mode],
                    dim=dim,
                )

                dict_new_mlps.append(dict_new_mlp)

        else:

            dict_new_flows.append(copy.deepcopy(dict_flows[i][mode]))

        dict_flows.append(dict_new_flows)

        if train_mlps:

            dict_mlps.append(dict_new_mlps)

        if mixture and update_weights:
            init_weights = model.weights

    to_return = {
        'xs': xs,
        'us': us,
        'accs': accs,
        'isomers': isomers,
        'time_mcmc': time_mcmc,
        'time_step_flow': time_step_flow,
        'time_step_adaptive': time_step_adaptive,
        'xs_proposals': xs_proposals,
        'us_proposals': us_proposals,
        'isomers_proposals': isomers_proposals,
        'dict_flows': dict_flows,
    }

    if use_calc:
            
        to_return['xs_calc'] = xs_calc
        to_return['us_calc'] = us_calc
        to_return['isomers_calc'] = isomers_calc
        to_return['inds_calc'] = inds_calc

    if train_mlps:

        to_return['dict_mlps'] = dict_mlps

    return to_return
    

In [20]:
flow_train = [load_csv_file('is'+str(i)+'_flow_train.csv', get_path() + '/andersen/datasets')[:, :14]
              for i in range(2)]
flow_test = [load_csv_file('is'+str(i)+'_flow_test.csv', get_path() + '/andersen/datasets')[:, :14]
                for i in range(2)]

In [21]:
dict_flow_models = [ load_pickle_file('dict_flow_model_is'+str(i)+'.pkl', get_path() + '/andersen/models/') for i in range(2)]
dict_mlp_models = [load_pickle_file('dict_mlp_model_is'+str(i)+'.pkl', get_path() + '/andersen/models/') for i in range(2)]

In [22]:
n_runs = 10
n_chains = 10
n_steps = 30

flow_hyperparams = {'n_iter': 10,
    'lr': 1e-3,
    'batch_size': 200,
    'use_scheduler': True,
    'step_schedule': 10,

    'energy_type': 'mlp',
    'compute_part_ratio': 'false',
    'n_prop': 10,

    'save_splits': 10,
    'grad_clip': 1e4,
    }

mlp_hyperparams = {'n_iter': 10,
    'lr': 1e-3,
    'batch_size': 200,

    'use_scheduler': True,
    'step_schedule': 100,

    'save_splits': 10,
    }

In [23]:
adaptive_sampling(mcmc_init=init, 
                n_chains=n_chains,
                n_steps=n_steps,
                n_runs=n_runs,
                flow_init_train=flow_train,
                flow_init_test=flow_test,
                dict_flows_init=dict_flow_models,
                flow_hyperparams=[flow_hyperparams, flow_hyperparams],
                energy_type='emt-mlp',
                temperature=350,
                mixture=True,
                dim=12,
                dict_mlps_init=dict_mlp_models,
                mlp_init_train=flow_train,
                mlp_init_test=flow_test,
                mlp_hyperparams=[mlp_hyperparams, mlp_hyperparams],
                train_mlps=True,
                frac_computed=0.2,
                init_weights=None,
                update_weights=True,
                scheduler_weights=10,
                n_samples_train_flow=None,
                folder_name='',
                alpha=0.5,
                device='cpu',
)

Adaptive sampling
Number of runs:  10
Number of chains:  10
Number of steps:  30
Temperature:  350
Energy type:  emt-mlp
Mixture model:  True
Isomer labels:  [0, 1]
Flow train dataset shapes:  [[4000, 14], [4000, 14]]
Flow test dataset shapes:  [[1000, 14], [1000, 14]]
tensor([4000, 4000])
tensor([200, 200])
tensor([20, 20], dtype=torch.int32)
MLP train dataset shapes:  [[4000, 14], [4000, 14]]
MLP test dataset shapes:  [[1000, 14], [1000, 14]]
Initial weights:  [0.5, 0.5]
Number of samples for training flows:  tensor([4000, 4000])
Current weights:  [0.5, 0.5]
Running Metropolis-Hastings
Number of chains: 10
Number of steps: 30
Temperature: 350K
Energy Type: emt-mlp
Mixture model
Use Neural Predictor: True
Use EMT Calculator: True
Mixture Model: True
Step 	 Acc Rate 	 Population
0 	 0.400 		 0.100
1 	 0.500 		 0.100
2 	 0.600 		 0.100
3 	 0.300 		 0.000
4 	 0.100 		 0.000
5 	 0.200 		 0.000
6 	 0.100 		 0.100
7 	 0.100 		 0.100
8 	 0.200 		 0.100
9 	 0.100 		 0.000
10 	 0.100 		 0.000


{'xs': [[tensor([[ 3.5307e-01,  8.4281e-02, -2.0749e-02, -1.0516e-01,  6.0574e-02,
            -7.9699e-02, -1.8674e-01,  1.6666e-01,  1.8222e-02, -2.7872e-01,
             4.8655e-01,  3.7278e-01],
           [ 1.8582e-01,  1.9943e-02,  1.0417e-01,  8.1872e-02,  6.0194e-03,
             2.0327e-02, -5.3821e-02,  8.6096e-02, -3.5873e-02, -5.0931e-02,
             7.0706e-02, -3.8785e-01],
           [-4.9108e-02, -1.1589e-01, -1.0551e-01, -9.8167e-02,  3.4588e-02,
             1.4554e-01,  5.4703e-02,  5.4388e-02,  3.8420e-03, -1.2027e-01,
             8.4034e-01,  1.0973e-01],
           [-1.2001e-01,  2.7320e-02, -1.5032e-01, -1.5417e-01, -2.9566e-02,
             7.9726e-02,  4.2423e-02,  8.0992e-02, -7.1277e-02, -5.1727e-02,
            -1.9828e-01, -2.2147e-01],
           [ 9.2860e-03,  1.2678e-02,  8.7172e-02, -8.9720e-02,  1.1166e-01,
             3.4998e-02,  3.3101e-02, -5.6621e-02, -2.9016e-02,  2.3713e-01,
             5.4354e-02, -8.4855e-02],
           [-1.5579e-02,  2.7